# Loan_approved Status - Model Building using KNN

# **Classification**

In [127]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [128]:
df = pd.read_csv(r"C:\Users\ksowm\Downloads\loan_approved.csv")
df

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status (Approved)
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


In [129]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Loan_ID                 614 non-null    object 
 1   Gender                  601 non-null    object 
 2   Married                 611 non-null    object 
 3   Dependents              599 non-null    object 
 4   Education               614 non-null    object 
 5   Self_Employed           582 non-null    object 
 6   ApplicantIncome         614 non-null    int64  
 7   CoapplicantIncome       614 non-null    float64
 8   LoanAmount              592 non-null    float64
 9   Loan_Amount_Term        600 non-null    float64
 10  Credit_History          564 non-null    float64
 11  Property_Area           614 non-null    object 
 12  Loan_Status (Approved)  614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [130]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [131]:
df.shape

(614, 13)

In [132]:
df.rename(columns = {"Loan_Status (Approved)":"Loan_Status_Approved"}, inplace = True)

In [133]:
# Loan_Status is not depends on Loan_ID and dependents , so drop those columns
df.drop(['Loan_ID'],axis = 1,inplace = True)

In [134]:
df.shape

(614, 12)

# Basic Check

In [135]:
df.isnull().sum()

Gender                  13
Married                  3
Dependents              15
Education                0
Self_Employed           32
ApplicantIncome          0
CoapplicantIncome        0
LoanAmount              22
Loan_Amount_Term        14
Credit_History          50
Property_Area            0
Loan_Status_Approved     0
dtype: int64

In [136]:
df.duplicated().sum()

0

In [137]:
df.columns

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area',
       'Loan_Status_Approved'],
      dtype='object')

# Outliers Detecting

In [138]:
num_cols = df.select_dtypes(include = ["int64","float64"]).columns
cat_cols = df.select_dtypes(exclude = ["int64","float64"]).columns

In [139]:
num_cols

Index(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History'],
      dtype='object')

In [140]:
cat_cols

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Property_Area', 'Loan_Status_Approved'],
      dtype='object')

In [141]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    print(col)
    print("IQR:",IQR)
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR
    print(f"Lower_limit:{lower_limit}")
    print(f"Upper_limit:{upper_limit}")
    print("MAX:",df[col].max())
    print("MIN:",df[col].min())

    outliers = df[(df[col] < lower_limit) | (df[col] > upper_limit)]
    print(f"{col} outliers count:", outliers.shape[0])
    print("-"*40)

ApplicantIncome
IQR: 2917.5
Lower_limit:-1498.75
Upper_limit:10171.25
MAX: 81000
MIN: 150
ApplicantIncome outliers count: 50
----------------------------------------
CoapplicantIncome
IQR: 2297.25
Lower_limit:-3445.875
Upper_limit:5743.125
MAX: 41667.0
MIN: 0.0
CoapplicantIncome outliers count: 18
----------------------------------------
LoanAmount
IQR: 68.0
Lower_limit:-2.0
Upper_limit:270.0
MAX: 700.0
MIN: 9.0
LoanAmount outliers count: 39
----------------------------------------
Loan_Amount_Term
IQR: 0.0
Lower_limit:360.0
Upper_limit:360.0
MAX: 480.0
MIN: 12.0
Loan_Amount_Term outliers count: 88
----------------------------------------
Credit_History
IQR: 0.0
Lower_limit:1.0
Upper_limit:1.0
MAX: 1.0
MIN: 0.0
Credit_History outliers count: 89
----------------------------------------


# Selection of Target and predictors

In [142]:
y = df["Loan_Status_Approved"]

In [143]:
X = df.drop(["Loan_Status_Approved"],axis = 1)

In [144]:
X.columns

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area'],
      dtype='object')

# Train - Test Split 

In [145]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2) 
print(X_train.shape)
print(X_test.shape)

(491, 11)
(123, 11)


# Handling Null values 

In [146]:
cat_cols = X_train.select_dtypes(exclude=["int64","float64"]).columns
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns

- **Fill Categorical Columns (Use Mode)**

In [147]:
# Fill categorical nulls
for col in cat_cols:
    mode_val = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode_val)
    X_test[col] = X_test[col].fillna(mode_val)

# Fill numerical nulls
for col in num_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

- **Fill Numerical Columns**

In [148]:
X_train.isnull().sum()

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
dtype: int64

# Handling Outliers

In [149]:
num_cols = X_train.select_dtypes(include = ["int64","float64"]).columns
cat_cols = X_train.select_dtypes(exclude = ["int64","float64"]).columns

In [150]:
iqr_limits = {}

for col in num_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    iqr_limits[col] = (lower, upper)
    
    lower, upper = iqr_limits[col]
    X_train[col] = X_train[col].clip(lower, upper)
    X_test[col]  = X_test[col].clip(lower, upper)

    
    outliers = X_train[(X_train[col] < lower) | (X_train[col] > upper)]
    print(f"{col} outliers count:", outliers.shape[0])


ApplicantIncome outliers count: 0
CoapplicantIncome outliers count: 0
LoanAmount outliers count: 0
Loan_Amount_Term outliers count: 0
Credit_History outliers count: 0


In [151]:
X_train.isnull().sum()

Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
dtype: int64

In [152]:
print("num_cols:" , num_cols)
print("cat_cols:" , cat_cols)

num_cols: Index(['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History'],
      dtype='object')
cat_cols: Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Property_Area'],
      dtype='object')


In [153]:
print(X_test.select_dtypes(include="object").columns)

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'Property_Area'],
      dtype='object')


# Apply Column Transformer for encoding and Scaling 

In [157]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
cat_cols = X_train.select_dtypes(include="object").columns
num_cols = X_train.select_dtypes(exclude="object").columns

preprocessor = ColumnTransformer(transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                                                ("num", StandardScaler(), num_cols)])


from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("knn", KNeighborsClassifier())])


In [154]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()

X_train[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test[cat_cols] = encoder.transform(X_test[cat_cols])

In [155]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [104]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Applied GridSearchCV 

In [158]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    verbose=1
)
grid.fit(X_train, y_train)


print("Best Parameters:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

final_model = grid.best_estimator_

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters: {'knn__n_neighbors': 11, 'knn__p': 2, 'knn__weights': 'uniform'}
Best CV accuracy: 0.6680478251906823


# Prediction

In [123]:
y_pred = final_model.predict(X_test)

# Evaluation

In [124]:
from sklearn.metrics import  accuracy_score
acc = accuracy_score(y_test,y_pred)
print("𝐓𝐞𝐬𝐭𝐢𝐧𝐠 𝐀𝐜𝐜𝐮𝐫𝐚𝐜𝐲:",acc)

𝐓𝐞𝐬𝐭𝐢𝐧𝐠 𝐀𝐜𝐜𝐮𝐫𝐚𝐜𝐲: 0.6666666666666666


In [125]:
y_train_pred = final_model.predict(X_train)

In [126]:
acc = accuracy_score(y_train,y_train_pred)
print("𝐓𝐫𝐚𝐢𝐧𝐢𝐧𝐠 𝐀𝐜𝐜𝐮𝐫𝐚𝐜𝐲:",acc)

𝐓𝐫𝐚𝐢𝐧𝐢𝐧𝐠 𝐀𝐜𝐜𝐮𝐫𝐚𝐜𝐲: 0.6904276985743381
